# Model Comparison

This notebook compares Logistic Regression, Random Forest, and Gradient Boosting using the Phase 4 training metadata and saved model artifacts. It reports cross-validation stability, validation metrics, untouched-test metrics, ROC curves, metric bars, and calibration proxies without selecting final risk thresholds.

In [1]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import RocCurveDisplay
ROOT = Path.cwd()
while not (ROOT / 'ml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.preprocessing.build_pipeline import PIPELINE_FILENAME
from ml.preprocessing.clean_data import TARGET_COLUMN
from ml.training.train_models import MODEL_FILENAMES, _classification_metrics
ARTIFACTS = ROOT / 'ml' / 'artifacts'
PROCESSED = ROOT / 'ml' / 'data' / 'processed'
sns.set_theme(style='whitegrid')

## Load artifacts and prepare validation/test data
The models and preprocessing transformer are the exact artifacts produced by Phase 4 and Phase 3.

In [ ]:
metadata = json.loads((ARTIFACTS / 'training_metadata.json').read_text())
preprocessor = joblib.load(ARTIFACTS / PIPELINE_FILENAME)
validation = pd.read_csv(PROCESSED / 'validation.csv')
test = pd.read_csv(PROCESSED / 'test.csv')
x_validation = preprocessor.transform(validation.drop(columns=[TARGET_COLUMN]))
x_test = preprocessor.transform(test.drop(columns=[TARGET_COLUMN]))
y_validation = validation[TARGET_COLUMN].astype(int)
y_test = test[TARGET_COLUMN].astype(int)
models = {name: joblib.load(ARTIFACTS / filename) for name, filename in MODEL_FILENAMES.items()}
print({name: type(model).__name__ for name, model in models.items()})

## Cross-validation summary
These values come from the selected hyperparameter candidate in the Phase 4 three-fold stratified CV search.

In [ ]:
cv_rows = []
for name, details in metadata['models'].items():
    cv_rows.append({'model': name, **details['cross_validation'], **details['validation_metrics']})
cv_summary = pd.DataFrame(cv_rows).set_index('model')
display(cv_summary[['best_cv_roc_auc', 'cv_roc_auc_std', 'roc_auc', 'f1', 'recall', 'brier_score']].round(4))

## Validation and untouched-test comparison

In [ ]:
rows = []
probabilities = {}
for name, model in models.items():
    validation_probability = model.predict_proba(x_validation)[:, 1]
    test_probability = model.predict_proba(x_test)[:, 1]
    probabilities[name] = (validation_probability, test_probability)
    for split_name, target, values in [('validation', y_validation, validation_probability), ('test', y_test, test_probability)]:
        rows.append({'model': name, 'split': split_name, **_classification_metrics(target, values)})
comparison = pd.DataFrame(rows)
display(comparison.set_index(['model', 'split']).round(4))

## ROC and metric visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, (validation_probability, _) in probabilities.items():
    RocCurveDisplay.from_predictions(y_validation, validation_probability, name=name, ax=axes[0])
axes[0].set_title('Validation ROC curves')
plot_data = comparison[comparison['split'] == 'validation'].melt(id_vars='model', value_vars=['roc_auc', 'f1', 'recall'], var_name='metric', value_name='value')
sns.barplot(data=plot_data, x='metric', y='value', hue='model', ax=axes[1])
axes[1].set_ylim(0, 1)
axes[1].set_title('Validation metric comparison')
plt.tight_layout()
plt.show()

## Calibration proxy comparison
Brier score and reliability curves are diagnostic evidence only. No model is treated as statistically calibrated until Phase 5 validates that decision.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
for name, (validation_probability, _) in probabilities.items():
    observed, predicted = calibration_curve(y_validation, validation_probability, n_bins=10, strategy='quantile')
    brier = cv_summary.loc[name, 'brier_score']
    ax.plot(predicted, observed, marker='o', label=f'{name} (Brier {brier:.3f})')
ax.set(xlabel='Mean predicted probability', ylabel='Observed default rate', title='Validation reliability curves')
ax.legend()
plt.show()

## Observations
- Compare ROC-AUC, F1, minority-class recall, Brier score, and cross-validation spread together; accuracy alone is insufficient for this imbalanced target.
- A small CV standard deviation indicates more stable validation ranking across folds, but it does not prove production calibration.
- The final primary model and dataset-derived risk thresholds must be selected in Phase 5 using the untouched test evidence and documented trade-offs.